# Célula 1 — Markdown
"""
### 3.2 Streaming Evaluation with SimulStream

Para simular condições de tradução simultânea (simultaneous speech translation),
utilizamos a biblioteca SimulStream, que emula o comportamento de um sistema
que precisa começar a traduzir antes de receber o áudio completo.

Métricas adicionais nesse cenário:
- Average Lagging (AL): atraso médio em relação ao orador
- Differentiable Average Lagging (DAL)
"""

In [ ]:
# Célula 2 — Simulação de streaming
from simuleval.evaluator import SentenceLevelEvaluator
# adapte conforme a API atual do SimulStream/SimulEval

resultados_streaming = []

for amostra in subset.select(range(30)):
    audio = amostra['audio']['array']
    sr    = amostra['audio']['sampling_rate']
    chunk_size_ms = 320  # 320ms por chunk — padrão SimulEval
    chunk_samples = int(sr * chunk_size_ms / 1000)
    
    chunks = [audio[i:i+chunk_samples] 
              for i in range(0, len(audio), chunk_samples)]
    
    parciais, timestamps = [], []
    t_inicio = time.perf_counter()
    
    for idx, chunk in enumerate(chunks):
        t_chunk = time.perf_counter()
        inputs = processor_seamless(
            audios=torch.tensor(chunk).unsqueeze(0).float(),
            sampling_rate=sr, src_lang="por", tgt_lang="eng",
            return_tensors="pt"
        )
        with torch.no_grad():
            out = model_seamless.generate(**inputs, tgt_lang="eng")
        texto = processor_seamless.decode(out[0].tolist(), skip_special_tokens=True)
        
        if texto.strip():
            parciais.append(texto)
            timestamps.append(time.perf_counter() - t_inicio)
    
    resultados_streaming.append({
        "modelo":    "SeamlessM4T-v2",
        "n_chunks":  len(chunks),
        "n_outputs": len(parciais),
        "latencia_primeiro_output_s": timestamps[0] if timestamps else None,
        "referencia": amostra['translation']
    })

df_stream = pd.DataFrame(resultados_streaming)
print(df_stream.describe())

In [ ]:
# Célula 3 — Figura 3: latência de primeiro output (Figure 3 do artigo)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_stream['latencia_primeiro_output_s'].dropna(),
        marker='o', markersize=4, linewidth=1, color='steelblue')
ax.axhline(df_stream['latencia_primeiro_output_s'].mean(),
           color='red', linestyle='--', label='Média')
ax.set_xlabel("Amostra")
ax.set_ylabel("Latência até 1º output (s)")
ax.set_title("Latência de primeiro output — modo streaming (SeamlessM4T)")
ax.legend()
plt.tight_layout()
plt.savefig("figures/fig3_streaming_latency.png", dpi=150)
plt.show()